# Building Retrieval Agents On Databricks

## Initial setup

In [0]:
%sql
-- setup to catalog and schema of studies
use catalog `studies`;
use schema `databricks_rag`;

select current_catalog() as actual_catalog,  current_schema() as actual_schema;

actual_catalog,actual_schema
studies,databricks_rag


In [0]:
catalog = 'studies'
schema = 'databricks_rag'

In [0]:
%sh
pip freeze | grep langchain

databricks-langchain==0.9.0
langchain==1.2.10
langchain-classic==1.0.1
langchain-community==0.4.1
langchain-core==1.2.13
langchain-experimental==0.3.4
langchain-mcp-adapters==0.2.1
langchain-openai==1.1.10
langchain-text-splitters==1.1.0
unitycatalog-langchain==0.3.0


In [0]:
from rich import print
import mlflow

# enable mlflow tracing
mlflow.langchain.autolog(disable=False)

In [0]:
SERVING_MODELS = {
    'gpt-5-1': 'databricks-gpt-5-1',  # disabled
    'gpt-oss-20b': 'databricks-gpt-oss-20b',  # disabled
    'meta-llama-8b': 'databricks-meta-llama-3-1-8b-instruct',  # enabled
    'qwen-80b': 'databricks-qwen3-next-80b-a3b-instruct',  # enabled
    'llama-maverick-400b': 'databricks-llama-4-maverick', # enabled
    'gemma-12b': 'databricks-gemma-3-12b'  # enabled  
}

ENDPOINT_LLM = SERVING_MODELS['qwen-80b']

VS_INDEX_NAME = 'studies.databricks_rag.docs_chunked_index'

# thread_id identifier
config = {'configurable': {'thread_id': 'databricks-build-rag-tests'}}

## Create an agent

In [0]:
from databricks.vector_search.client import VectorSearchClient

index_name = f'{catalog}.{schema}.docs_chunked_index'
vs_client = VectorSearchClient(disable_notice=True)
vector_store = vs_client.get_index(index_name=index_name)

In [0]:
from langchain.agents import create_agent
from langchain_core.tools import tool
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.chat_models import ChatDatabricks

llm_model = ChatDatabricks(
	endpoint=ENDPOINT_LLM,
	max_tokens='300',
 	verbose=True
)


@tool
def retrieve_context(query_text: str):
    '''Search PDTIC internal documents'''
    retrieved_docs = vector_store.similarity_search(
        query_text=query_text,
        columns=['path', 'chunk'],
        num_results=2
    )
    return retrieved_docs

tools = [retrieve_context]

checkpointer = InMemorySaver()

system_prompt = '''Answer the questions using only internal documents. Include references if possible. Do not make assumptions outside the provided context.
'''

agent = create_agent(
    model=llm_model,
    tools=tools,
    system_prompt=system_prompt
)

/home/spark-76250368-b610-47b0-bef5-a8/.ipykernel/3712/command-8046825959173185-3367043533:6: LangChainDeprecationWarning: The class `ChatDatabricks` was deprecated in LangChain 0.3.3 and will be removed in 1.0. An updated version of the class exists in the `databricks-langchain package and should be used instead. To use it run `pip install -U `databricks-langchain` and import as `from `databricks_langchain import ChatDatabricks``.
  llm_model = ChatDatabricks(


In [0]:
user_content = 'Quais as diretrizes recomendadas pelo PDTIC?'
user_input= {'messages': [{'role': 'user', 'content': user_content}]}

res = agent.invoke(
    input=user_input,
    config=config,
    verbose=True,
    handle_parsing_error=True
)

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Trace(trace_id=tr-ec2d786e832ca274af0a9049b1834fa6)

In [0]:
print(res['messages'][-1].content)

As diretrizes recomendadas pelo PDTIC estão relacionadas à gestão da força de trabalho e dos recursos de Tecnologia
da Informação e Comunicação (TIC), com o objetivo de orientar a atuação institucional no período de 2024 a 2025. O 
plano é desenvolvido em colaboração com a CTIC e o CGGD, e a Diretoria de Pesquisas e Informações Estratégicas 
(DIE) atua como unidade técnica responsável por sua elaboração.

Para obter um detalhamento completo das diretrizes, é necessário consultar o documento completo do Plano Diretor de
Tecnologia da Informação e Comunicação (PDTIC) 2024-2025.

## Log agent to model registry

In [0]:

import yaml

def create_config(endpoint_model: str, vs_index_name: str, num_results: int) -> dict:
	config_yaml = {
		'endpoint_model': endpoint_model,
		'vector_search': {
			'index_name': vs_index_name,
			'num_results': num_results
		}
	}
	
	return config_yaml

In [0]:
agent_config = create_config(endpoint_model=ENDPOINT_LLM, vs_index_name=VS_INDEX_NAME, num_results=2)

# export yaml agent config
with open('agent_config.yaml', 'w', encoding='utf-8') as file:
	file.write(yaml.safe_dump(agent_config))
	
print(f'# agent_config.yaml\n{yaml.safe_dump(agent_config)}')

# agent_config.yaml
endpoint_model: databricks-qwen3-next-80b-a3b-instruct
vector_search:
  index_name: studies.databricks_rag.docs_chunked_index
  num_results: 2

## Export agent code as `agent.py`

In [0]:
%%writefile agent.py

from uuid import uuid4
import mlflow
import mlflow.pyfunc
from typing import List
from pydantic import BaseModel
from langchain_core.tools import tool
from langchain.agents import create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain_community.chat_models import ChatDatabricks
from databricks.vector_search.client import VectorSearchClient


SERVING_MODELS = {
    'gpt-5-1': 'databricks-gpt-5-1',  # disabled
    'gpt-oss-20b': 'databricks-gpt-oss-20b',  # disabled
    'meta-llama-8b': 'databricks-meta-llama-3-1-8b-instruct',  # enabled
    'qwen-80b': 'databricks-qwen3-next-80b-a3b-instruct',  # enabled
    'llama-maverick-400b': 'databricks-llama-4-maverick', # enabled
    'gemma-12b': 'databricks-gemma-3-12b'  # enabled  
}

ENDPOINT_LLM = SERVING_MODELS['qwen-80b']

VS_INDEX_NAME = 'studies.databricks_rag.docs_chunked_index'

# thread_id identifier
config = {'configurable': {'thread_id': 'databricks-build-rag-tests'}}

catalog = 'studies'
schema = 'databricks_rag'
index_name = f'{catalog}.{schema}.docs_chunked_index'
vs_client = VectorSearchClient(disable_notice=True)
vector_store = vs_client.get_index(index_name=index_name)


class Message(BaseModel):
    role: str
    content: str


class AgentInput(BaseModel):
    messages: List[Message]


class AgentWrapper(mlflow.pyfunc.PythonModel):

    def __init__(self, vector_store, llm_endpoint: str):
        self.vector_store = vector_store
        self.llm_endpoint = llm_endpoint        

    def load_context(self, context):
        '''Recreates the agent in the serving environment.'''
        llm = ChatDatabricks(
            endpoint=self.llm_endpoint,
            max_tokens=300,
            verbose=False
        )

        @tool
        def retrieve_context(query_text: str):
            '''Search PDTIC internal documents'''
            return vector_store.similarity_search(
                query_text=query_text,
                columns=['path', 'chunk'],
                num_results=2
            )

        checkpointer = InMemorySaver()

        self.agent = create_agent(
            model=llm,
            tools=[retrieve_context],
            checkpointer=checkpointer
        )

    def predict(
        self,
        context: mlflow.pyfunc.PythonModelContext,
        model_input: List[AgentInput]
    ) -> List[str]:

        outputs: List[str] = []

        for item in model_input:
            response = self.agent.invoke(
                item.model_dump(),
                config={
                    'configurable': {
                        'thread_id': f'serving_thread-{uuid4()}',
                        'checkpoint_ns': 'mlflow_serving'
                    }
                }
            )

            outputs.append(response['messages'][-1].content)

        return outputs

Overwriting agent.py


## Log model in MLflow

In [0]:
from agent import AgentWrapper, AgentInput, Message

agent_wrapper = AgentWrapper(
    llm_endpoint=ENDPOINT_LLM,
    vector_store=vector_store
)

input_example = [
    AgentInput(messages=[Message(role='user', content='Quais as diretrizes recomendadas pelo PDTIC?')])
]

# run and log
with mlflow.start_run():
    mlflow.pyfunc.log_model(
        name='AgentSearchPDTIC',
        python_model=agent_wrapper,
        input_example=input_example,
        registered_model_name='studies.databricks_rag.ModelAgentSearchPDTIC'
    )

🔗 View Logged Model at: https://dbc-34ac7b3c-7a54.cloud.databricks.com/ml/experiments/1523098827396138/models/m-ba0907bdda0447aaa52d520f0ef6e092?o=7474656785376246
2026/02/20 09:27:37 INFO mlflow.models.signature: Running the predict function to generate output based on input example


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


2026/02/20 09:27:52 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Registered model 'studies.databricks_rag.ModelAgentSearchPDTIC' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/11 [00:00<?, ?it/s]

🔗 Created version '33' of model 'studies.databricks_rag.modelagentsearchpdtic': https://dbc-34ac7b3c-7a54.cloud.databricks.com/explore/data/models/studies/databricks_rag/modelagentsearchpdtic/version/33?o=7474656785376246


## Run a version of the agent

In [0]:
AGENT_VERSION = 20

loaded_model = mlflow.pyfunc.load_model(
    f'models:/studies.databricks_rag.ModelAgentSearchPDTIC/{AGENT_VERSION}'
)

res = loaded_model.predict([
    AgentInput(messages=[Message(role='user', content='Para que serve o planejamento TIC?')])
])
print(res[0])

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


O planejamento de TIC (Tecnologia da Informação e Comunicação) serve como um processo de gestão norteador para a 
execução das ações e projetos de TIC dentro de uma organização. Seu principal objetivo é conferir foco à atuação da
área de TIC, apresentando estratégias e traçando planos de ação para implementá-las. Isso permite o direcionamento 
eficiente de esforços e recursos, garantindo que as iniciativas de TIC estejam alinhadas aos objetivos estratégicos
da organização.

Além disso, o planejamento de TIC também envolve atividades como monitorar sua execução, avaliar resultados e 
realizar ajustes quando necessário, assegurando sua eficácia contínua.

Trace(trace_id=tr-f4ae07098620d8d869b1548b86138b04)

In [0]:
input_example = [
    AgentInput(messages=[Message(role='user', content='Quais as diretrizes recomendadas pelo PDTIC?')])
]

with mlflow.start_run():
    agent_logged_infos = mlflow.pyfunc.log_model(
        name='AgentSearchPDTIC',
        python_model=agent_wrapper,
        code_paths=['agent.py', 'agent_config.yaml'],
        input_example=input_example,
        registered_model_name='studies.databricks_rag.ModelAgentSearchPDTIC'
    )

🔗 View Logged Model at: https://dbc-34ac7b3c-7a54.cloud.databricks.com/ml/experiments/1523098827396138/models/m-a09311bcb6964d9ca21901b001646dc2?o=7474656785376246
2026/02/20 09:28:11 INFO mlflow.models.signature: Running the predict function to generate output based on input example


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


2026/02/20 09:28:23 WARNING mlflow.utils.requirements_utils: Found pyspark version (4.0.0+databricks.connect.17.3.2) contains a local version label (+databricks.connect.17.3.2). MLflow logged a pip requirement for this package as 'pyspark==4.0.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.


Registered model 'studies.databricks_rag.ModelAgentSearchPDTIC' already exists. Creating a new version of this model...


Uploading artifacts:   0%|          | 0/13 [00:00<?, ?it/s]

🔗 Created version '34' of model 'studies.databricks_rag.modelagentsearchpdtic': https://dbc-34ac7b3c-7a54.cloud.databricks.com/explore/data/models/studies/databricks_rag/modelagentsearchpdtic/version/34?o=7474656785376246


In [0]:
print(f'agent_logged_infos.model_uri: {agent_logged_infos.model_uri}')

agent_logged_infos.model_uri: models:/m-a09311bcb6964d9ca21901b001646dc2

## Register agent model into Unity Catalog 

In [0]:
mlflow.set_registry_uri(catalog)
model_uri = agent_logged_infos.model_uri
uc_model_name = f'{catalog}.{schema}.ModelAgentSearchPDTIC'

# register
agent_registered_infos = mlflow.register_model(
    model_uri=model_uri,
    name=uc_model_name
)

Registered model 'studies.databricks_rag.ModelAgentSearchPDTIC' already exists. Creating a new version of this model...
Created version '5' of model 'studies.databricks_rag.ModelAgentSearchPDTIC'.


In [0]:
print(f'Registered agent version: {agent_registered_infos.version}')

Registered agent version: 5

<img src="./images/7. model registered.png">

## Test inference with registered agent model

In [0]:
pyfunc_model = mlflow.pyfunc.load_model(model_uri)
input_example = pyfunc_model.input_example
print(f'Input example: {input_example}')

res = mlflow.models.predict(
    model_uri=model_uri,
    input_data=input_example,
    # env_manager='uv'
)

Input example: [{'messages': [{'role': 'user', 'content': 'Quais as diretrizes recomendadas pelo PDTIC?'}]}]

2026/02/20 09:28:39 INFO mlflow.models.python_api: It is highly recommended to use `uv` as the environment manager for predicting with MLflow models as its performance is significantly better than other environment managers. Run `pip install uv` to install uv. See https://docs.astral.sh/uv/getting-started/installation for other installation methods.


2026/02/20 09:28:40 INFO mlflow.models.flavor_backend_registry: Selected backend for flavor 'python_function'


2026/02/20 09:28:42 INFO mlflow.utils.virtualenv: Installing python 3.12.3 if it does not exist
-> https://www.python.org/ftp/python/3.12.3/Python-3.12.3.tar.xz
Installing Python-3.12.3...
Installed Python-3.12.3 to /tmp/pyenv_root/versions/3.12.3
2026/02/20 09:31:33 INFO mlflow.utils.virtualenv: Creating a new environment in /tmp/virtualenv_envs/mlflow-a50ffed7ad6d143bfa41ff2e93aa402921ec63b7 with /tmp/pyenv_root/versions/3.12.3/bin/python
2026/02/20 09:31:33 INFO mlflow.utils.virtualenv: Installing dependencies


created virtual environment CPython3.12.3.final.0-64 in 252ms
  creator CPython3Posix(dest=/tmp/virtualenv_envs/mlflow-a50ffed7ad6d143bfa41ff2e93aa402921ec63b7, clear=False, no_vcs_ignore=False, global=False)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/home/spark-76250368-b610-47b0-bef5-a8/.local/share/virtualenv)
    added seed packages: pip==25.0.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 32.4 MB/s eta 0:00:00



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 158.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 116.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 119.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 109.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 160.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.2/15.2 MB 191.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 MB 176.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.1/38.1 MB 151.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 152.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 61.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 7


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
2026/02/20 09:33:06 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-a50ffed7ad6d143bfa41ff2e93aa402921ec63b7/bin/activate && python -c ""']'
2026/02/20 09:33:06 INFO mlflow.utils.environment: === Running command '['bash', '-c', 'source /tmp/virtualenv_envs/mlflow-a50ffed7ad6d143bfa41ff2e93aa402921ec63b7/bin/activate && python /local_disk0/.ephemeral_nfs/envs/pythonEnv-76250368-b610-47b0-bef5-a884d9ee8551/lib/python3.12/site-packages/mlflow/pyfunc/_mlflow_pyfunc_backend_predict.py --model-uri file:///local_disk0/user_tmp_data/spark-76250368-b610-47b0-bef5-a8/tmpo24qjr55 --content-type json --input-path /local_disk0/user_tmp_data/spark-76250368-b610-47b0-bef5-a8/tmpk5pjw7sq/input.json']'
Fri Feb 20 09:33:11 2026 Connection to spark using Py4J from PID  17565
Fri Feb 20 09:33:11 2026 Initialized gateway on port 46771
Fri Fe

[NOTICE] Using a notebook authentication token. Recommended for development only. For improved performance, please use Service Principal based authentication. To disable this message, pass disable_notice=True.
{"predictions": ["As diretrizes recomendadas pelo PDTIC (Plano Diretor de Tecnologia da Informa\u00e7\u00e3o e Comunica\u00e7\u00e3o) est\u00e3o detalhadas no documento oficial para o per\u00edodo de 2024 a 2025. Embora os trechos recuperados n\u00e3o apresentem todas as diretrizes de forma completa, eles indicam que o PDTIC tem como objetivo orientar a gest\u00e3o da for\u00e7a de trabalho e dos recursos de TIC, em colabora\u00e7\u00e3o com a CTIC (Comiss\u00e3o de Tecnologia da Informa\u00e7\u00e3o e Comunica\u00e7\u00e3o) e o CGGD (Coordenadoria Geral de Gest\u00e3o de Dados).\n\nA Diretoria de Pesquisas e Informa\u00e7\u00f5es Estrat\u00e9gicas (DIE) \u00e9 a unidade t\u00e9cnica respons\u00e1vel por sua implementa\u00e7\u00e3o. Para obter um conjunto completo e detalhado das

<img src="./images/8. artifacts of the agent model.png">